# COMP663 Assignment 2 — Task 3: Bayesian Optimisation

This notebook runs Task 3 only. It uses the same stratified 60/20/20 split, scaler rule, baseline architecture, seed, and 20-epoch training budget as Task 2.

### 3.1 Hyperparameters to optimise

Bayesian optimisation searches learning rate, batch size, and weight decay: the same three training hyperparameters searched in Task 2. The supplied baseline architecture remains fixed.

### 3.2 Objective, search space, surrogate model, and acquisition process

The objective is validation macro-F1. Five initial configurations are sampled randomly. A Gaussian Process Regressor with a Matern kernel models completed trials. Learning rate and weight decay are represented on log scales; batch size is categorical and represented with one-hot encoding. Expected Improvement combines the predicted macro-F1 and uncertainty to choose each of the next seven candidate configurations.

### 3.3 Evaluation procedure and primary metric

Every trial trains on the same 60% training split. Macro-F1 and balanced accuracy are calculated only on the same 20% validation split. The scaler is fitted on the training split only to prevent data leakage.

### 3.4 Computational budget and justification

The budget is fixed at 12 trials of 20 epochs: five random initial trials and seven Gaussian Process plus Expected Improvement trials. This gives Task 3 the same search space and total trial budget as Task 2 for a fair comparison.


In [ ]:
from pathlib import Path
import random
import time

import numpy as np
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn

setup_started = time.perf_counter()
SEED = 42
SEARCH_EPOCHS = 20
BAYESIAN_TRIALS = 12

if Path("/kaggle").exists() and not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU was not allocated.")
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    RUNTIME_DEVICE = f"{torch.cuda.get_device_name(0)} x{torch.cuda.device_count()}"
elif DEVICE.type == "mps":
    RUNTIME_DEVICE = "Apple Silicon MPS"
else:
    RUNTIME_DEVICE = "CPU"

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

def format_decimal(value):
    return np.format_float_positional(float(value), unique=True, trim="-")

def log_cell(name, started, configuration):
    elapsed = format_decimal(time.perf_counter() - started)
    print(f"{name}: device={RUNTIME_DEVICE}; configuration={configuration}; elapsed_seconds={elapsed}")

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
KAGGLE_DATA_PATH = Path("/kaggle/input/datasets/yangliunz/comp663-a2-forest-cove/forest_cover_data.csv")
kaggle_data_paths = ([KAGGLE_DATA_PATH] if KAGGLE_DATA_PATH.exists() else []) + list(Path("/kaggle/input").rglob("forest_cover_data.csv"))
DATA_PATH = kaggle_data_paths[0] if kaggle_data_paths else ROOT / "data" / "forest_cover_data.csv"
if not DATA_PATH.exists() and Path("/kaggle").exists():
    import kagglehub
    DATA_PATH = next(Path(kagglehub.dataset_download("yangliunz/comp663-a2-forest-cover")).rglob("forest_cover_data.csv"))
log_cell("Environment setup", setup_started, f"seed={SEED}, data={DATA_PATH}, trials={BAYESIAN_TRIALS}")


In [ ]:
data_started = time.perf_counter()
data = pd.read_csv(DATA_PATH).dropna(subset=["Cover_Type"])
target = "Cover_Type"
feature_names = [column for column in data.columns if column != target]
continuous_features = [column for column in feature_names if not column.startswith("Wilderness_Area")]
assert data.shape == (571_012, 15), data.shape

train_validation_frame, test_frame = train_test_split(data, test_size=0.20, stratify=data[target], random_state=SEED)
train_frame, validation_frame = train_test_split(train_validation_frame, test_size=0.25, stratify=train_validation_frame[target], random_state=SEED)
scaler = StandardScaler().fit(train_frame[continuous_features])

def prepare_data(frame):
    features = frame[feature_names].astype("float32").copy()
    features[continuous_features] = scaler.transform(features[continuous_features])
    return features.to_numpy(), frame[target].to_numpy(dtype=np.int64) - 1

x_train, y_train = prepare_data(train_frame)
x_validation, y_validation = prepare_data(validation_frame)
x_train_tensor = torch.tensor(x_train, dtype=torch.float32, device=DEVICE)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
x_validation_tensor = torch.tensor(x_validation, dtype=torch.float32, device=DEVICE)

class BaselineNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 24), nn.Sigmoid(),
            nn.Linear(24, 12), nn.Sigmoid(),
            nn.Linear(12, 5),
        )

    def forward(self, x):
        return self.layers(x)

def train_and_evaluate(config):
    torch.manual_seed(SEED)
    model = BaselineNN(len(feature_names)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()
    generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    model.train()
    for _ in range(config["epochs"]):
        order = torch.randperm(len(x_train_tensor), generator=generator, device=DEVICE)
        for start in range(0, len(order), config["batch_size"]):
            batch_index = order[start : start + config["batch_size"]]
            optimizer.zero_grad()
            loss = loss_fn(model(x_train_tensor[batch_index]), y_train_tensor[batch_index])
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(x_validation_tensor), dim=1)
    prediction = probabilities.argmax(dim=1).cpu().numpy()
    return f1_score(y_validation, prediction, average="macro"), balanced_accuracy_score(y_validation, prediction)

log_cell("Data and baseline setup", data_started, "split=0.6/0.2/0.2, architecture=14-24-12-5 Sigmoid, scaler=StandardScaler")


### 3.5 Apply Bayesian optimisation


In [3]:
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern

# Record the total Bayesian-optimisation time.
search_started = time.perf_counter()
bayesian_rows = []
# Encode the categorical batch-size choices for the Gaussian Process.
BATCH_SIZES = np.array([256, 512, 1024])
LOG_LR_BOUNDS = (-4.0, -2.0)
LOG_WEIGHT_DECAY_BOUNDS = (-5.0, -3.0)


# Expected Improvement balances predicted score and uncertainty.
def expected_improvement(mu, sigma, best_so_far, xi=0.01):
    # We maximise macro-F1, so improvement means predicting ABOVE best_so_far.
    sigma = np.maximum(sigma, 0.000000001)
    improvement = mu - best_so_far - xi
    z = improvement / sigma
    return improvement * norm.cdf(z) + sigma * norm.pdf(z)


# Scale continuous variables and one-hot encode batch size for GP input.
def gp_features(points):
    points = np.asarray(points)
    lower = np.array([LOG_LR_BOUNDS[0], LOG_WEIGHT_DECAY_BOUNDS[0]])
    upper = np.array([LOG_LR_BOUNDS[1], LOG_WEIGHT_DECAY_BOUNDS[1]])
    continuous = (points[:, :2] - lower) / (upper - lower)
    batch_one_hot = np.eye(len(BATCH_SIZES))[points[:, 2].astype(int)]
    return np.column_stack([continuous, batch_one_hot])


# Convert a candidate into training settings and evaluate one validation trial.
def evaluate_configuration(log_learning_rate, log_weight_decay, batch_index, trial_number):
    config = {
        "learning_rate": float(10 ** log_learning_rate),
        "batch_size": int(BATCH_SIZES[int(batch_index)]),
        "weight_decay": float(10 ** log_weight_decay),
        "epochs": SEARCH_EPOCHS,
    }
    print(f"Task 3 trial {trial_number}/{BAYESIAN_TRIALS}: learning_rate={format_decimal(config['learning_rate'])}, batch_size={config['batch_size']}, weight_decay={format_decimal(config['weight_decay'])}, epochs={config['epochs']}", flush=True)
    started = time.perf_counter()
    macro_f1, balanced_accuracy = train_and_evaluate(config)
    row = {"trial": trial_number, **config, "macro_f1": macro_f1, "balanced_accuracy": balanced_accuracy, "parameters": 725, "seconds": time.perf_counter() - started}
    bayesian_rows.append(row)
    print(f"Task 3 trial {trial_number} result: macro-F1={format_decimal(macro_f1)}, balanced accuracy={format_decimal(balanced_accuracy)}, seconds={format_decimal(row['seconds'])}", flush=True)
    return macro_f1


# Evaluate five random initial points before fitting the surrogate.
rng = np.random.default_rng(SEED)
initial_points = np.column_stack([
    rng.uniform(*LOG_LR_BOUNDS, 5),
    rng.uniform(*LOG_WEIGHT_DECAY_BOUNDS, 5),
    rng.integers(0, len(BATCH_SIZES), 5),
])
evaluated_points = list(initial_points)
evaluated_scores = [evaluate_configuration(point[0], point[1], point[2], trial_number) for trial_number, point in enumerate(initial_points, 1)]

# Define the Tutorial 5 Gaussian Process surrogate and candidate grid.
kernel = ConstantKernel(1.0, (0.01, 100.0)) * Matern(length_scale=0.3, length_scale_bounds=(0.05, 3.0), nu=2.5)
log_learning_rates = np.linspace(*LOG_LR_BOUNDS, 40)
log_weight_decays = np.linspace(*LOG_WEIGHT_DECAY_BOUNDS, 40)
learning_rate_grid, weight_decay_grid, batch_grid = np.meshgrid(log_learning_rates, log_weight_decays, np.arange(len(BATCH_SIZES)), indexing="ij")
candidates = np.column_stack([learning_rate_grid.ravel(), weight_decay_grid.ravel(), batch_grid.ravel()])
candidate_used = np.zeros(len(candidates), dtype=bool)

# Fit the GP after each result and select the candidate with the highest EI.
for trial_number in range(6, BAYESIAN_TRIALS + 1):
    gp = GaussianProcessRegressor(kernel=kernel, alpha=0.001, normalize_y=True, n_restarts_optimizer=2, random_state=SEED)
    gp.fit(gp_features(evaluated_points), evaluated_scores)
    mean, standard_deviation = gp.predict(gp_features(candidates), return_std=True)
    improvement = expected_improvement(mean, standard_deviation, max(evaluated_scores))
    improvement[candidate_used] = -np.inf
    next_index = int(np.argmax(improvement))
    next_point = candidates[next_index]
    candidate_used[next_index] = True
    evaluated_points.append(next_point)
    evaluated_scores.append(evaluate_configuration(next_point[0], next_point[1], next_point[2], trial_number))

# Sort completed trials so the best validation macro-F1 appears first.
bayesian_table = pd.DataFrame(bayesian_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
best_bayesian = bayesian_table.iloc[0]
print("Task 3 complete:")
print(f"Best Bayesian configuration: learning_rate={format_decimal(best_bayesian['learning_rate'])}, batch_size={int(best_bayesian['batch_size'])}, weight_decay={format_decimal(best_bayesian['weight_decay'])}, epochs={int(best_bayesian['epochs'])}")
print(bayesian_table.to_string(index=False, float_format=lambda value: format_decimal(value)))
log_cell("Task 3 Bayesian optimisation", search_started, f"method=Gaussian Process plus Expected Improvement, batch_size=one-hot categorical, trials={BAYESIAN_TRIALS}, epochs_per_trial={SEARCH_EPOCHS}")


Task 3 trial 1/12: learning_rate=0.00353111691382141, batch_size=512, weight_decay=0.0008938089586343595, epochs=20
Task 3 trial 1 result: macro-F1=0.36765088688041786, balanced accuracy=0.3539646519785102, seconds=23.80778444300006
Task 3 trial 2/12: learning_rate=0.000754669641079693, batch_size=512, weight_decay=0.0003328736391557839, epochs=20
Task 3 trial 2 result: macro-F1=0.4017970264372613, balanced accuracy=0.38409752960236354, seconds=17.67033995700001
Task 3 trial 3/12: learning_rate=0.005214297905300546, batch_size=256, weight_decay=0.0003733607072503032, epochs=20
Task 3 trial 3 result: macro-F1=0.42119014945993083, balanced accuracy=0.4019880579276133, seconds=35.2360818840001
Task 3 trial 4/12: learning_rate=0.002481624443014984, batch_size=1024, weight_decay=0.000018039615030067847, epochs=20
Task 3 trial 4 result: macro-F1=0.5144614784334609, balanced accuracy=0.47769149705260305, seconds=8.987462680000021
Task 3 trial 5/12: learning_rate=0.00015429601005519726, batch_

### 3.6 Best configuration, performance, and search time

| Item | Value |
|---|---:|
| Learning rate | 0.01 |
| Batch size | 512 |
| Weight decay | 0.00001 |
| Epochs | 20 |
| Validation macro-F1 | 0.620486164019339 |
| Validation balanced accuracy | 0.5615715029443511 |
| Search time on Tesla T4 x2 (seconds) | 228.18862507699998 |

### 3.7 Search results table

The first five trials were random initial configurations. GP/EI then explored promising regions across all three hyperparameters. Trial 9 selected learning rate = 0.01, batch size = 512, and weight decay = 0.00001, which achieved the best macro-F1. Trials 7 and 8 showed that changing batch size to 1024 in the same high-learning-rate, low-weight-decay region produced lower macro-F1 values of 0.5628973313639289 and 0.5609506222558817.
